# AI/BI Adoption Dashboard - Metadata Collection

This notebook collects metadata from various Databricks AI/BI services using the Databricks SDK.

## Tables Created

This notebook creates the following Delta tables:

### Genie
* `adb_genie_spaces` - All Genie spaces
* `adb_genie_conversations` - All conversations within Genie spaces
* `adb_genie_messages` - All messages with query results and errors

### Dashboards
* `adb_dashboards` - All Lakeview dashboards
* `adb_dashboard_schedules` - Dashboard schedules
* `adb_dashboard_subscriptions` - Dashboard subscriptions

### Models & Serving
* `adb_models` - Unity Catalog registered models
* `adb_serving_endpoints` - Model serving endpoints

### Apps
* `adb_apps` - Databricks Apps

## Parameters
* `catalog_name` - Target catalog for tables
* `schema_name` - Target schema for tables

In [0]:
%pip install databricks_sdk --upgrade

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog_name", "users")
dbutils.widgets.text("schema_name", "")
dbutils.widgets.text("skip_get_conversations", "false")

skip_get_conversations = dbutils.widgets.get("skip_get_conversations").lower() == 'true'
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
assert catalog_name and schema_name, "catalog_name and schema_name must be provided"


In [0]:
from databricks.sdk import WorkspaceClient
from datetime import datetime
import pandas as pd
import json
import pyspark.sql.functions as F 

w = WorkspaceClient()

In [0]:
# List of all tables to drop
table_names = [
    "adb_genie_spaces",
    "adb_genie_conversations",
    "adb_genie_messages",
    "adb_dashboards",
    "adb_dashboard_schedules",
    "adb_dashboard_subscriptions",
    "adb_models",
    "adb_serving_endpoints",
    "adb_apps"
]

print("Dropping existing tables if they exist...")
for table_name in table_names:
    full_table_name = f"{catalog_name}.{schema_name}.{table_name}"
    try:
        spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
        print(f"✓ Dropped table: {full_table_name}")
    except Exception as e:
        print(f"⚠ Could not drop {full_table_name}: {e}")

print(f"\nCompleted deletion process for {len(table_names)} tables.")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, LongType

# Define schemas for all tables
schemas = {
    "adb_genie_spaces": StructType([
        StructField('space_id', StringType(), True),
        StructField('name', StringType(), True),
        StructField('description', StringType(), True),
        StructField('warehouse_id', StringType(), True)
    ]),
    
    "adb_genie_conversations": StructType([
        StructField('space_id', StringType(), True),
        StructField('conversation_id', StringType(), True),
        StructField('created_timestamp', TimestampType(), True),
        StructField('title', StringType(), True),
        StructField('user_id', LongType(), True)
    ]),
    
    "adb_genie_messages": StructType([
        StructField('space_id', StringType(), True),
        StructField('space_name', StringType(), True),
        StructField('message_id', StringType(), True),
        StructField('conversation_id', StringType(), True),
        StructField('user_id', StringType(), True),
        StructField('user_email', StringType(), True),
        StructField('status', StringType(), True),
        StructField('created_timestamp', TimestampType(), True),
        StructField('last_updated_timestamp', TimestampType(), True),
        StructField('user_question', StringType(), True),
        StructField('ai_response', StringType(), True),
        StructField('sql_query', StringType(), True),
        StructField('statement_id', StringType(), True),
        StructField('suggested_questions', StringType(), True),
        StructField('num_attachments', IntegerType(), True),
        StructField('feedback_rating', StringType(), True),
        StructField('error_type', StringType(), True),
        StructField('error_message', StringType(), True)
    ]),
    
    "adb_dashboards": StructType([
        StructField('dashboard_id', StringType(), True),
        StructField('display_name', StringType(), True),
        StructField('create_time', TimestampType(), True),
        StructField('lifecycle_state', StringType(), True),
        StructField('update_time', TimestampType(), True),
        StructField('warehouse_id', StringType(), True)
    ]),
    
    "adb_dashboard_schedules": StructType([
        StructField('dashboard_id', StringType(), True),
        StructField('schedule_id', StringType(), True),
        StructField('create_time', StringType(), True),
        StructField('display_name', StringType(), True),
        StructField('pause_status', StringType(), True)
    ]),
    
    "adb_dashboard_subscriptions": StructType([
        StructField('dashboard_id', StringType(), True),
        StructField('schedule_id', StringType(), True),
        StructField('subscription_id', StringType(), True),
        StructField('create_time', TimestampType(), True),
        StructField('user_id', StringType(), True),
        StructField('destination_id', StringType(), True)
    ]),
    
    "adb_models": StructType([
        StructField('full_name', StringType(), True),
        StructField('name', StringType(), True),
        StructField('catalog_name', StringType(), True),
        StructField('schema_name', StringType(), True),
        StructField('created_at', TimestampType(), True),
        StructField('created_by', StringType(), True),
        StructField('updated_at', TimestampType(), True),
        StructField('updated_by', StringType(), True),
        StructField('owner', StringType(), True),
        StructField('comment', StringType(), True),
        StructField('num_users_with_access', IntegerType(), True),
        StructField('num_groups_with_access', IntegerType(), True)
    ]),
    
    "adb_serving_endpoints": StructType([
        StructField('name', StringType(), True),
        StructField('id', StringType(), True),
        StructField('creation_timestamp', TimestampType(), True),
        StructField('creator', StringType(), True),
        StructField('last_updated_timestamp', TimestampType(), True),
        StructField('state', StringType(), True),
        StructField('models', StringType(), True),
        StructField('num_users_with_access', IntegerType(), True),
        StructField('num_groups_with_access', IntegerType(), True)
    ]),
    
    "adb_apps": StructType([
        StructField('name', StringType(), True),
        StructField('id', StringType(), True),
        StructField('create_time', TimestampType(), True),
        StructField('creator', StringType(), True),
        StructField('update_time', TimestampType(), True),
        StructField('updater', StringType(), True),
        StructField('url', StringType(), True),
        StructField('app_status', StringType(), True),
        StructField('compute_status', StringType(), True),
        StructField('num_users_with_access', IntegerType(), True),
        StructField('num_groups_with_access', IntegerType(), True)
    ])
}

# Create all empty tables
print("Initializing empty tables...")
for table_name, schema in schemas.items():
    full_table_name = f"{catalog_name}.{schema_name}.{table_name}"
    empty_df = spark.createDataFrame([], schema)
    empty_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table_name)
    print(f"✓ Created empty table: {full_table_name}")

print(f"\nAll {len(schemas)} tables initialized successfully!")

## Genie Spaces, Conversations, and Messages

This section collects metadata about Genie spaces and their usage:
* **adb_genie_spaces**: All Genie spaces in the workspace
* **adb_genie_conversations**: All conversations within each Genie space
* **adb_genie_messages**: All messages within each conversation, including query results and errors

In [0]:
spaces = []
page_token = None

# w = WorkspaceClient()

while True:
    response = w.genie.list_spaces(page_token=page_token)
    for s in response.spaces:
        spaces.append({
            "space_id": getattr(s, "space_id", None),
            "name": getattr(s, "title", None),
            "description": getattr(s, "description", None),
            "warehouse_id": getattr(s, "warehouse_id", None)
        })
    if not response.next_page_token or response.next_page_token == "":
        break
    page_token = response.next_page_token

pdf = pd.DataFrame(spaces)
if not pdf.empty:
    genie_spaces_df = spark.createDataFrame(pdf)
    genie_spaces_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{catalog_name}.{schema_name}.adb_genie_spaces"
    )
    print(f"Loaded {genie_spaces_df.count()} Genie spaces into table adb_genie_spaces")
else:
    print("No Genie spaces found.")

In [0]:
from databricks.sdk import WorkspaceClient
from datetime import datetime
import pandas as pd

#w = WorkspaceClient()

conversations = []
has_manage_on_all_spaces = True
include_all_conversations = has_manage_on_all_spaces

# Check if the adb_genie_spaces table exists before proceeding
if not spark.catalog.tableExists(f"{catalog_name}.{schema_name}.adb_genie_spaces"):
    print("No Genie spaces table found. Skipping conversation collection.")
else:
    # Get all space IDs from the previously created table
    space_ids = [
        row.space_id
        for row in spark.table(f"{catalog_name}.{schema_name}.adb_genie_spaces").select('space_id').collect()
    ]
    
    if not space_ids:
        print("No Genie spaces found in table. Skipping conversation collection.")
    else:
        print(f"Fetching conversations for {len(space_ids)} Genie spaces...")

        for space_id in space_ids:
            try:
                page_token = None
                while True:
                    response = w.genie.list_conversations(space_id=space_id, include_all=include_all_conversations, page_token=page_token)
                    for conv in response.conversations:
                        conv_dict = conv.as_dict()
                        conv_dict['space_id'] = space_id
                        conversations.append(conv_dict)

                    if not response.next_page_token or response.next_page_token == "":
                        break
                    page_token = response.next_page_token
                    
            except Exception as e:
                print(f"Error fetching conversations for space {space_id}: {e}")
                continue

if conversations:
    genie_conversations_df = spark.createDataFrame(conversations).withColumn('created_timestamp', F.from_unixtime(F.col('created_timestamp')/1000).cast('timestamp'))
    genie_conversations_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{catalog_name}.{schema_name}.adb_genie_conversations"
    )
    print(f"Loaded {genie_conversations_df.count()} Genie conversations into table adb_genie_conversations")
else:
    print("No Genie conversations found.")

In [0]:
from typing import Optional, List, Dict, Any
from databricks.sdk.service.iam import User

# Cache for user ID to email mappings
USER_CACHE = {}

def _resolve_user_email(user_id: Optional[str], w: WorkspaceClient) -> Optional[str]:
    """Resolves a user ID to an email address using the SCIM Users API."""
    if not user_id or user_id in ("None", "0"):
        return None
    
    if user_id in USER_CACHE:
        return USER_CACHE[user_id]
    
    try:
        user: User = w.users.get(user_id)
        email = user.user_name
        USER_CACHE[user_id] = email
        return email
    except Exception:
        # Return a placeholder if user lookup fails
        return f"ID_{user_id}"

def _extract_message_data(message, space_id: str, space_name: str, w: WorkspaceClient) -> Dict[str, Any]:
    """Extracts relevant fields from a GenieMessage into a flat dictionary."""
    msg_dict = message.as_dict()
    
    # Extract IDs (API has both 'id' and 'message_id' for legacy compatibility)
    msg_id = msg_dict.get('message_id') or msg_dict.get('id')
    user_id = msg_dict.get('user_id')
    user_email = _resolve_user_email(str(user_id) if user_id else None, w)

    record = {
        'space_id': space_id,
        'space_name': space_name,
        'message_id': msg_id,
        'conversation_id': msg_dict.get('conversation_id'),
        'user_id': str(user_id) if user_id else None,
        'user_email': user_email,
        'status': str(msg_dict.get('status')) if msg_dict.get('status') else None,
        'created_timestamp': msg_dict.get('created_timestamp'),
        'last_updated_timestamp': msg_dict.get('last_updated_timestamp'),
        'user_question': msg_dict.get('content'),
    }
    
    # Process attachments
    ai_responses, sql_queries, statement_ids, suggested_qs = [], [], [], []
    attachments = msg_dict.get('attachments') or []
    
    for att in attachments:
        # Text attachment
        if text_obj := att.get('text'):
            ai_responses.append(text_obj.get('content', ''))
        
        # Query attachment
        if query_obj := att.get('query'):
            sql_queries.append(query_obj.get('query', ''))
            if stmt_id := query_obj.get('statement_id'):
                statement_ids.append(str(stmt_id))
        
        # Suggested questions attachment
        if sq_obj := att.get('suggested_questions'):
            suggested_qs.extend(sq_obj.get('questions', []))

    def join_non_empty(items: List, sep: str = ' | ') -> Optional[str]:
        filtered = list(filter(None, items))
        return sep.join(filtered) if filtered else None

    record.update({
        'ai_response': join_non_empty(ai_responses),
        'sql_query': join_non_empty(sql_queries),
        'statement_id': join_non_empty(statement_ids),
        'suggested_questions': join_non_empty(suggested_qs, ', '),
        'num_attachments': len(attachments),
    })
    
    # Feedback rating
    feedback = msg_dict.get('feedback') or {}
    record['feedback_rating'] = str(feedback.get('rating')) if feedback.get('rating') else 'NONE'
    
    # Error info
    error = msg_dict.get('error')
    if error and isinstance(error, dict):
        record['error_type'] = error.get('type')
        record['error_message'] = error.get('message') or error.get('error')
    elif error:
        record['error_type'] = 'Unknown'
        record['error_message'] = str(error)
    else:
        record['error_type'] = None
        record['error_message'] = None
    
    return record

messages = []

# Check if required tables exist before proceeding
if not spark.catalog.tableExists(f"{catalog_name}.{schema_name}.adb_genie_spaces"):
    print("No Genie spaces table found. Skipping message collection.")
elif not spark.catalog.tableExists(f"{catalog_name}.{schema_name}.adb_genie_conversations"):
    print("No Genie conversations table found. Skipping message collection.")
else:
    # Get all space_id, space_name, and conversation_id from the previously created tables
    conversation_data = [
        (row.space_id, row.space_name, row.conversation_id)
        for row in spark.sql(f"""
            SELECT c.space_id, s.name as space_name, c.conversation_id
            FROM {catalog_name}.{schema_name}.adb_genie_conversations c
            INNER JOIN {catalog_name}.{schema_name}.adb_genie_spaces s
            ON c.space_id = s.space_id
        """).collect()
    ]
    
    if not conversation_data:
        print("No conversations found in table. Skipping message collection.")
    else:
        print(f"Fetching messages for {len(conversation_data)} conversations...")

        for space_id, space_name, conversation_id in conversation_data:
            if skip_get_conversations:
                print('Skipping conversations pull as skip_get_conversations is set to true')
                break
            try:
                page_token = None
                while True:
                    response = w.genie.list_conversation_messages(
                        space_id=space_id,
                        conversation_id=conversation_id,
                        page_token=page_token
                    )
                    if not response.messages:
                        break
                    
                    for msg in response.messages:
                        message_data = _extract_message_data(msg, space_id, space_name, w)
                        messages.append(message_data)
                    
                    if not response.next_page_token or response.next_page_token == "":
                        break
                    page_token = response.next_page_token
                    
            except Exception as e:
                print(f"Error fetching messages for conversation {conversation_id} in space {space_id}: {e}")
                continue

if messages:
    genie_messages_df = spark.createDataFrame(
        messages,
        'space_id string, space_name string, message_id string, conversation_id string, user_id string, user_email string, status string, created_timestamp bigint, last_updated_timestamp bigint, user_question string, ai_response string, sql_query string, statement_id string, suggested_questions string, num_attachments int, feedback_rating string, error_type string, error_message string'
    ) \
        .withColumn('created_timestamp', F.from_unixtime(F.col('created_timestamp')/1000).cast('timestamp')) \
        .withColumn('last_updated_timestamp', F.from_unixtime(F.col('last_updated_timestamp')/1000).cast('timestamp'))
    genie_messages_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{catalog_name}.{schema_name}.adb_genie_messages"
    )
    print(f"Loaded {genie_messages_df.count()} Genie messages into table adb_genie_messages")
else:
    print("No Genie messages found or process skipped.")

## Lakeview Dashboards, Schedules, and Subscriptions

This section collects metadata about Lakeview dashboards and their distribution:
* **adb_dashboards**: All Lakeview dashboards in the workspace
* **adb_dashboard_schedules**: Scheduled refresh/distribution for dashboards
* **abd_dashboard_subscriptions**: User and destination subscriptions for dashboard schedules

In [0]:
from databricks.sdk import WorkspaceClient
from datetime import datetime

#w = WorkspaceClient()

rows = []
for d in w.lakeview.list(page_size=100):
    d_dict = d.as_dict()
    # Convert timestamps to Python datetime if they are not already
    create_time = d_dict.get("create_time")
    update_time = d_dict.get("update_time")
    if isinstance(create_time, str):
        try:
            create_time = datetime.fromisoformat(create_time)
        except Exception:
            create_time = None
    if isinstance(update_time, str):
        try:
            update_time = datetime.fromisoformat(update_time)
        except Exception:
            update_time = None
    rows.append((
        d_dict.get("dashboard_id"),
        d_dict.get("display_name"),
        create_time,
        d_dict.get("lifecycle_state"),
        update_time,
        d_dict.get("warehouse_id")
    ))

from pyspark.sql.types import StructType, StructField, StringType, TimestampType
schema = StructType([
    StructField('dashboard_id', StringType(), True),
    StructField('display_name', StringType(), True),
    StructField('create_time', TimestampType(), True),
    StructField('lifecycle_state', StringType(), True),
    StructField('update_time', TimestampType(), True),
    StructField('warehouse_id', StringType(), True)
])

df = spark.createDataFrame(
    rows,
    schema
)
df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.{schema_name}.adb_dashboards")
#display(df)

In [ ]:
from databricks.sdk import WorkspaceClient
from datetime import datetime

schedules = []

dashboard_ids = [
    row.dashboard_id
    for row in spark.table(f"{catalog_name}.{schema_name}.adb_dashboards").select("dashboard_id").collect()
]

for dashboard_id in dashboard_ids:
    try:
        for sched in w.lakeview.list_schedules(dashboard_id=dashboard_id):
            sched_dict = sched.as_dict()
            subscriber = sched_dict.get("subscriber", {})
            destination_subscriber = subscriber.get("destination_subscriber")
            user_subscriber = subscriber.get("user_subscriber")
            schedules.append({
                "dashboard_id": dashboard_id,
                "schedule_id": sched_dict.get("schedule_id"),
                "create_time": sched_dict.get("create_time"),
                "display_name": sched_dict.get("display_name"),
                "pause_status": sched_dict.get("pause_status")
            })
    except Exception as e:
        # Optionally log or print the dashboard_id that failed
        continue

if schedules:
    import pandas as pd
    pdf_sched = pd.DataFrame(schedules)
    spark_df_sched = spark.createDataFrame(pdf_sched)
    spark.sql(f"DROP TABLE IF EXISTS {catalog_name}.{schema_name}.adb_dashboard_schedules")
    spark_df_sched.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.{schema_name}.adb_dashboard_schedules")
    #display(spark_df_sched)
else:
    print("No dashboard schedules found. Ensuring empty table exists.")
    spark.createDataFrame([], schemas["adb_dashboard_schedules"]).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.{schema_name}.adb_dashboard_schedules")

In [0]:
# Databricks notebook Python
from databricks.sdk import WorkspaceClient
import pandas as pd

source_table = f"{catalog_name}.{schema_name}.adb_dashboards"
schedule_table = f"{catalog_name}.{schema_name}.adb_dashboard_schedules"
target_table = f"{catalog_name}.{schema_name}.adb_dashboard_subscriptions"

# Check if source tables exist
if not spark.catalog.tableExists(schedule_table):
    print(f"Table {schedule_table} not found. Skipping subscription extraction.")
else:
    #w = WorkspaceClient()

    # ---------- helpers ----------
    def get_field(obj_or_dict, *names):
        """Return the first non-None among possible field names (works for dicts or objects)."""
        for n in names:
            if isinstance(obj_or_dict, dict):
                if n in obj_or_dict and obj_or_dict[n] is not None:
                    return obj_or_dict[n]
            else:
                v = getattr(obj_or_dict, n, None)
                if v is not None:
                    return v
        return None

    def to_dict_safe(x):
        return (getattr(x, "as_dict", lambda: {})() or {}) if x is not None else {}

    # ---------- get dashboard ids ----------
    sched_df = spark.table(schedule_table).collect()
    rows = []
    count = 0

    for r in sched_df:
        dashboard_id = r["dashboard_id"]
        schedule_id = r["schedule_id"]
        try:
            # enumerate subscriptions for this schedule
            for sub in w.lakeview.list_subscriptions(dashboard_id=dashboard_id, schedule_id=schedule_id):
                subdict = to_dict_safe(sub)
                subscriber = get_field(subdict, "subscriber") or get_field(sub, "subscriber") or {}

                user_subscriber = get_field(subscriber, "user_subscriber")
                destination_subscriber = get_field(subscriber, "destination_subscriber")

                user_id = (
                    get_field(user_subscriber or {}, "user_id", "id")
                    if user_subscriber is not None else None
                )

                subscription_id = get_field(subdict, "subscription_id") or get_field(sub, "subscription_id")
                create_time = get_field(subdict, "create_time") or get_field(sub, "create_time")

                # capture destination info if present
                destination_type = get_field(destination_subscriber or {}, "destination_type", "type")
                destination_id = get_field(destination_subscriber or {}, "destination_id", "destination", "id")

                rows.append({
                    "dashboard_id": dashboard_id,
                    "schedule_id": schedule_id,
                    "subscription_id": subscription_id,
                    "create_time": create_time,
                    "user_id": user_id,
                    "destination_id": destination_id,
                })
                count += 1

        except Exception as e:
            continue

    print(f"Collected {count} subscription rows across dashboards.")

    # ---------- write to Delta table ----------
    pdf = pd.DataFrame(rows)

    if pdf.empty:
        spark_df = spark.createDataFrame([], schemas["adb_dashboard_subscriptions"])
    else:
        spark_df = spark.createDataFrame(pdf)

    spark.sql(f"DROP TABLE IF EXISTS {target_table}")

    from pyspark.sql import functions as F
    spark_df = (
        spark_df
        .withColumn("create_time", F.to_timestamp("create_time"))
        .select("dashboard_id", "schedule_id", "subscription_id", "create_time", "user_id", "destination_id")
    )

    spark_df.write.format("delta").mode("overwrite").saveAsTable(target_table)
    print(f"Successfully created {target_table}")

## Unity Catalog Models

This section collects metadata about registered models in Unity Catalog:
* **adb_models**: All registered models with permissions and metadata

In [0]:
from databricks.sdk import WorkspaceClient
from datetime import datetime
import pandas as pd

#w = WorkspaceClient()

models = []

try:
    for model in w.registered_models.list():
        model_dict = model.as_dict() if hasattr(model, 'as_dict') else {}
        
        # Get permissions summary
        num_users_with_access = 0
        num_groups_with_access = 0
        try:
            full_name = model_dict.get('full_name') or f"{model_dict.get('catalog_name', '')}.{model_dict.get('schema_name', '')}.{model_dict.get('name', '')}"
            if full_name and full_name != '..':
                perms = w.registered_models.get_permissions(full_name)
                if perms and hasattr(perms, 'access_control_list'):
                    acl = perms.access_control_list or []
                    for entry in acl:
                        if hasattr(entry, 'user_name') and entry.user_name:
                            num_users_with_access += 1
                        if hasattr(entry, 'group_name') and entry.group_name:
                            num_groups_with_access += 1
        except Exception as e:
            # Permissions may not be accessible, continue without them
            pass
        
        # Convert timestamps
        created_at = model_dict.get('created_at')
        updated_at = model_dict.get('updated_at')
        if isinstance(created_at, (int, float)) and created_at > 0:
            try:
                created_at = datetime.fromtimestamp(created_at / 1000)
            except Exception:
                created_at = None
        if isinstance(updated_at, (int, float)) and updated_at > 0:
            try:
                updated_at = datetime.fromtimestamp(updated_at / 1000)
            except Exception:
                updated_at = None
        
        models.append({
            "full_name": model_dict.get('full_name'),
            "name": model_dict.get('name'),
            "catalog_name": model_dict.get('catalog_name'),
            "schema_name": model_dict.get('schema_name'),
            "created_at": created_at,
            "created_by": model_dict.get('created_by'),
            "updated_at": updated_at,
            "updated_by": model_dict.get('updated_by'),
            "owner": model_dict.get('owner'),
            "comment": model_dict.get('comment'),
            "num_users_with_access": num_users_with_access,
            "num_groups_with_access": num_groups_with_access
        })
except Exception as e:
    print(f"Error listing models: {e}")

pdf = pd.DataFrame(models)
if not pdf.empty:
    spark_df = spark.createDataFrame(pdf)
    spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{catalog_name}.{schema_name}.adb_models"
    )
    print(f"Loaded {spark_df.count()} models into table adb_models")
else:
    print("No models found.")

## Model Serving Endpoints

This section collects metadata about model serving endpoints:
* **adb_serving_endpoints**: All serving endpoints with their models and permissions

In [0]:
from databricks.sdk import WorkspaceClient
from datetime import datetime
import pandas as pd

#w = WorkspaceClient()

serving_endpoints = []

try:
    for endpoint in w.serving_endpoints.list():
        endpoint_dict = endpoint.as_dict() if hasattr(endpoint, 'as_dict') else {}
        
        # Get permissions summary
        num_users_with_access = 0
        num_groups_with_access = 0
        try:
            endpoint_name = endpoint_dict.get('name')
            if endpoint_name:
                perms = w.serving_endpoints.get_permissions(endpoint_name)
                if perms and hasattr(perms, 'access_control_list'):
                    acl = perms.access_control_list or []
                    for entry in acl:
                        if hasattr(entry, 'user_name') and entry.user_name:
                            num_users_with_access += 1
                        if hasattr(entry, 'group_name') and entry.group_name:
                            num_groups_with_access += 1
        except Exception as e:
            # Permissions may not be accessible, continue without them
            pass
        
        # Extract model info from config
        config = endpoint_dict.get('config', {})
        models_info = []
        if isinstance(config, dict):
            served_models = config.get('served_models', [])
            if isinstance(served_models, list):
                for model in served_models:
                    if isinstance(model, dict):
                        model_name = model.get('model_name') or model.get('name')
                        if model_name:
                            models_info.append(model_name)
        
        # Convert timestamps
        creation_timestamp = endpoint_dict.get('creation_timestamp')
        last_updated_timestamp = endpoint_dict.get('last_updated_timestamp')
        if isinstance(creation_timestamp, (int, float)) and creation_timestamp > 0:
            try:
                creation_timestamp = datetime.fromtimestamp(creation_timestamp / 1000)
            except Exception:
                creation_timestamp = None
        if isinstance(last_updated_timestamp, (int, float)) and last_updated_timestamp > 0:
            try:
                last_updated_timestamp = datetime.fromtimestamp(last_updated_timestamp / 1000)
            except Exception:
                last_updated_timestamp = None
        
        serving_endpoints.append({
            "name": endpoint_dict.get('name'),
            "id": endpoint_dict.get('id'),
            "creation_timestamp": creation_timestamp,
            "creator": endpoint_dict.get('creator'),
            "last_updated_timestamp": last_updated_timestamp,
            "state": endpoint_dict.get('state'),
            "models": ','.join(models_info) if models_info else None,
            "num_users_with_access": num_users_with_access,
            "num_groups_with_access": num_groups_with_access
        })
except Exception as e:
    print(f"Error listing serving endpoints: {e}")

pdf = pd.DataFrame(serving_endpoints)
if not pdf.empty:
    spark_df = spark.createDataFrame(pdf)
    spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{catalog_name}.{schema_name}.adb_serving_endpoints"
    )
    print(f"Loaded {spark_df.count()} serving endpoints into table adb_serving_endpoints")
else:
    print("No serving endpoints found.")


## Databricks Apps

This section collects metadata about Databricks Apps:
* **adb_apps**: All apps with their status and permissions

In [0]:
from databricks.sdk import WorkspaceClient
from datetime import datetime
import pandas as pd

#w = WorkspaceClient()

apps = []

try:
    for app in w.apps.list():
        app_dict = app.as_dict() if hasattr(app, 'as_dict') else {}
        
        # Get permissions summary
        num_users_with_access = 0
        num_groups_with_access = 0
        try:
            app_name = app_dict.get('name')
            if app_name:
                perms = w.apps.get_permissions(app_name)
                if perms and hasattr(perms, 'access_control_list'):
                    acl = perms.access_control_list or []
                    for entry in acl:
                        if hasattr(entry, 'user_name') and entry.user_name:
                            num_users_with_access += 1
                        if hasattr(entry, 'group_name') and entry.group_name:
                            num_groups_with_access += 1
        except Exception as e:
            # Permissions may not be accessible, continue without them
            pass
        
        # Convert timestamps
        create_time = app_dict.get('create_time')
        update_time = app_dict.get('update_time')
        if isinstance(create_time, str):
            try:
                create_time = datetime.fromisoformat(create_time.replace('Z', '+00:00'))
            except Exception:
                create_time = None
        if isinstance(update_time, str):
            try:
                update_time = datetime.fromisoformat(update_time.replace('Z', '+00:00'))
            except Exception:
                update_time = None
        
        apps.append({
            "name": app_dict.get('name'),
            "id": app_dict.get('id'),
            "create_time": create_time,
            "creator": app_dict.get('creator'),
            "update_time": update_time,
            "updater": app_dict.get('updater'),
            "url": app_dict.get('url'),
            "app_status": app_dict.get('app_status'),
            "compute_status": app_dict.get('compute_status'),
            "num_users_with_access": num_users_with_access,
            "num_groups_with_access": num_groups_with_access
        })
except Exception as e:
    print(f"Error listing apps: {e}")

pdf = pd.DataFrame(apps)
if not pdf.empty:
    spark_df = spark.createDataFrame(pdf)
    spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{catalog_name}.{schema_name}.adb_apps"
    )
    print(f"Loaded {spark_df.count()} apps into table adb_apps")
else:
    print("No apps found.")
